<a href="https://colab.research.google.com/github/rickyajs/Data-Science-2026/blob/main/Pertemuan_3_RickyArmandaJayaSirait_240401020219.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nama  : Ricky Armanda Jaya Sirait
# Nim   : 240401020219
# Kelas  : IF401
# Prodi  : PJJ S1 Informatika

In [12]:
import pandas as pd
import numpy as np
import requests
from scipy.stats.mstats import winsorize
from pandas import json_normalize

In [21]:
try:
    df = pd.read_csv('housing_dirty.csv')
except FileNotFoundError:
    print("Mengingatkan: File 'housing_dirty.csv' tidak ditemukan. Membuat data simulasi...")
    data_simulasi = {
        'id': [1, 2, 2, 3, 4, 5, 6],
        'luas_m2': [120, np.nan, np.nan, 85, 450, 110, 90],
        'harga_juta': [1200, 850, 850, np.nan, 25000, 950, 1100],
        'kota': ['jakarta ', ' Jakarta', ' Jakarta', 'bandung', 'surabaya', 'jakarta', 'medan'],
        'kamar': [3, 2, 2, np.nan, 5, 3, 2],
        'tahun_bangun': [2015, 2010, 2010, 2018, 1920, 2012, 2014],
        'kondisi': ['Baik', 'baik', 'baik', 'Butuh Renovasi', 'bagus', 'BAIK', 'renovasi']
    }
    df = pd.DataFrame(data_simulasi)

In [14]:
print('=== LOAD & EKSPLORASI AWAL ===')
print('Shape awal:', df.shape)
print('\nJumlah missing values per kolom:')
print(df.isnull().sum())
print("-" * 50)

=== LOAD & EKSPLORASI AWAL ===
Shape awal: (130, 7)

Jumlah missing values per kolom:
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64
--------------------------------------------------


In [15]:
print('=== HAPUS DUPLIKAT ===')
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)
print("-" * 50)

=== HAPUS DUPLIKAT ===
Setelah hapus duplikat: (130, 7)
--------------------------------------------------


In [16]:
print('=== NORMALISASI STRING ===')
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()
print("Normalisasi string selesai.")
print("-" * 50)

=== NORMALISASI STRING ===
Normalisasi string selesai.
--------------------------------------------------


In [17]:
print('=== IMPUTASI MISSING VALUES ===')
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])
print("Imputasi missing values selesai.")
print("-" * 50)

=== IMPUTASI MISSING VALUES ===
Imputasi missing values selesai.
--------------------------------------------------


In [18]:
print('=== TANGANI OUTLIER (IQR FENCE) ===')
# Melakukan perulangan untuk kolom numerik sesuai instruksi langkah 8 & template
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    # Menggunakan clip() untuk membatasi nilai ekstrem (capping) sesuai batas IQR
    df[col] = df[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
print("Penanganan outlier selesai.")
print("-" * 50)

=== TANGANI OUTLIER (IQR FENCE) ===
Penanganan outlier selesai.
--------------------------------------------------


In [19]:
print('=== VALIDASI & EKSPOR ===')
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print('Shape akhir:', df.shape)

df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan dengan nama: housing_clean.csv')
print("-" * 50)

=== VALIDASI & EKSPOR ===
Shape akhir: (130, 7)
Dataset bersih tersimpan dengan nama: housing_clean.csv
--------------------------------------------------


In [20]:
print('=== AKSES API JSONPLACEHOLDER ===')
URL = "https://jsonplaceholder.typicode.com/users"

try:
    # Mengakses API publik dengan timeout 10 detik untuk mencegah hanging
    response = requests.get(URL, timeout=10)

    # Memastikan response sukses (status code 200)
    if response.status_code == 200:
        data = response.json()

        # Mengonversi list JSON menjadi DataFrame, meratakan nested structure (seperti address/company) dengan separator '_'
        df_users = json_normalize(data, sep='_')

        print("Berhasil mengambil data dari API!")
        print(f"Shape DataFrame API: {df_users.shape}")
        print("\nMenampilkan beberapa kolom contoh (id, name, email, address_city):")
        # Menampilkan kolom tertentu seperti pada panduan modul halaman 10
        kolom_pilihan = ['id', 'name', 'email', 'address_city']
        if all(col in df_users.columns for col in kolom_pilihan):
            print(df_users[kolom_pilihan].head())
        else:
            print(df_users.iloc[:, :4].head())  # Fallback jika nama kolom nested berbeda

    else:
        print(f"Gagal mengambil data. Error Status Code: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"Terjadi error koneksi saat mengakses API: {e}")

print("=" * 50)


=== AKSES API JSONPLACEHOLDER ===
Berhasil mengambil data dari API!
Shape DataFrame API: (10, 15)

Menampilkan beberapa kolom contoh (id, name, email, address_city):
   id              name                      email   address_city
0   1     Leanne Graham          Sincere@april.biz    Gwenborough
1   2      Ervin Howell          Shanna@melissa.tv    Wisokyburgh
2   3  Clementine Bauch         Nathan@yesenia.net  McKenziehaven
3   4  Patricia Lebsack  Julianne.OConner@kory.org    South Elvis
4   5  Chelsey Dietrich   Lucio_Hettinger@annie.ca     Roscoeview
